[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/09_reviewer_response_synthesis.ipynb)

# Step 09 - Reviewer-response synthesis

This notebook integrates Steps 00-08 into reviewer-facing traceability, claim-maturity, manuscript asset tables, and selected reviewer-action gate audits. The selected-action audits create no new biological evidence and run no Step 04 optimizer; they derive objective gate tables from current outputs so unsupported claims can be restricted, upgraded, or kept blocked without silent overclaiming.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", ".")).resolve()
if not (PROJECT_ROOT / "src").exists():
    current = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [current, *current.parents] if (p / "src").exists()), PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


## Selected-action gate audits

In [ ]:
from src.reviewer_gate_audits import ReviewerGateAuditConfig, run_reviewer_gate_audits

gate_config = ReviewerGateAuditConfig(write_outputs=True)
gate_result = run_reviewer_gate_audits(PROJECT_ROOT, gate_config)
pd.DataFrame([
    {"artifact": key, "n_rows": len(value), "n_columns": len(value.columns)}
    for key, value in gate_result.items()
]).sort_values("artifact").reset_index(drop=True)


## Selected-action scientific value

In [ ]:
selected_summary = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "selected_action_results_summary.csv")
scientific_value = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "selected_action_scientific_value_assessment.csv")
display(selected_summary)
display(scientific_value)


In [ ]:
from src.step09_reviewer_synthesis import Step09Config, run_step09_reviewer_synthesis

config = Step09Config(write_outputs=True)
result = run_step09_reviewer_synthesis(PROJECT_ROOT, config)
result["analysis_summary"]

## R1-R7 traceability

In [ ]:
traceability = result["reviewer_traceability_table"]
traceability

## Claim maturity

In [ ]:
maturity = result["claim_maturity_table"]
maturity

## Manuscript asset manifest

In [ ]:
manifest = result["manuscript_asset_manifest"]
manifest

## Objective synthesis conclusion

The final row below is generated from the actual upstream outputs. If final biological degeneracy wording is not allowed, the manuscript response should keep the claim at mechanism-screen or prediction-limited maturity and state the unresolved layer explicitly.

In [ ]:
final_claim = maturity[maturity["claim"].eq("final biological degeneracy wording is allowed")]
display(final_claim)
assert set(traceability["reviewer_id"]) == {"R1", "R2", "R3", "R4", "R5", "R6", "R7"}
assert "final_biological_degeneracy_claim_allowed" in result["analysis_summary"]
print("Step 09 synthesis outputs written under", PROJECT_ROOT / "outputs" / "reviewer_synthesis")

## Restricted claim gates

In [ ]:
restricted_join = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "restricted_all_gate_join.csv")
restricted_claims = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "restricted_validation_claims.csv")
assumption_gate = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "assumption_gate_audit.csv")
notebook_screen = pd.read_csv(PROJECT_ROOT / "outputs" / "reviewer_synthesis" / "notebook_update_screen_after_selected_actions.csv")
display(restricted_join.groupby(["validation_label", "restricted_degeneracy_claim_allowed", "blocking_axes"], dropna=False).size().reset_index(name="n_rows"))
display(restricted_claims)
display(assumption_gate)
display(notebook_screen)


## Post-execution scientific status

Executed status for reviewer response after selected actions: Step 09 integrates R1-R7 into 7 traceability rows, 6 claim-maturity rows, and a complete expanded manuscript manifest. The selected gate audits produce no-refit/current-output evidence plus modest sensitivity screens. They raise the mechanism-regime claim only to `restricted_model_phenotype_support`, because some model-derived phenotype/stratum gates pass. They do **not** support global biological degeneracy: the restricted all-gate join has no passing row, the intracellular-K proxy assumption gate fails, and parameter semantics/identifiability block direct physiological parameter wording. Final biological degeneracy wording therefore remains `not_allowed_yet`.

Notebook-screen result: core Steps 05-08 do not require computational reruns because their source outputs were reused unchanged, but their interpretation comments can optionally point to the new derivative audits. Step 09 must be rerun whenever these selected-action gates change.